In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)

In [5]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
results

[{'output': '# Python function to extract AWS account ID from ARN\n\n```python\ndef extract_account_id_from_arn(arn: str) -> str | None:\n    """\n    Extracts the AWS account ID from an ARN string.\n    \n    Args:\n        arn: An AWS ARN string in the format:\n             arn:partition:service:region:account-id:resource-type/resource-id\n    \n    Returns:\n        The AWS account ID (12-digit string) if present in the ARN, None otherwise.\n    \n    Examples:\n        >>> extract_account_id_from_arn(\'arn:aws:iam::123456789012:user/john\')\n        \'123456789012\'\n        >>> extract_account_id_from_arn(\'arn:aws:s3:::my-bucket\')\n        None\n        >>> extract_account_id_from_arn(\'arn:aws:ec2:us-east-1:987654321098:instance/i-1234567890abcdef0\')\n        \'987654321098\'\n    """\n    if not isinstance(arn, str):\n        return None\n    \n    # ARN format: arn:partition:service:region:account-id:resource\n    parts = arn.split(\':\')\n    \n    # ARN must have at least 